In this dataset the unit is the user, since it's a pre-aggregated assignment table, one row per user with their variant and outcome, so I deduplicated at the user level and removed users with conflicting group assignments. If this were raw e-commerce clickstream, the unit would be different: I'd sessionize first (typically a 30-minute inactivity window defining a visit), decide whether the analysis unit is the visit or the user, and make sure randomization happened at the right level. Mismatching the randomization unit and the analysis unit is a classic A/B testing error.

In [2]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv('ab_data.csv')
data.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [4]:
print("Rows:", len(data))
print("\nGroup sizes:")
print(data['group'].value_counts())
print("\nLanding page counts:")
print(data['landing_page'].value_counts())

Rows: 294478

Group sizes:
group
treatment    147276
control      147202
Name: count, dtype: int64

Landing page counts:
landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64


In [5]:
# In a clean experiment: control should ALWAYS see old_page, treatment ALWAYS new_page.
# Let's check how often that's violated.
mismatch = data[
    ((data['group'] == 'control') & (data['landing_page'] == 'new_page')) |
    ((data['group'] == 'treatment') & (data['landing_page'] == 'old_page'))
]
print("Mismatched rows (group doesn't match page shown):", len(mismatch))
print("That's", f"{len(mismatch)/len(data)*100:.2f}%", "of the data")

Mismatched rows (group doesn't match page shown): 3893
That's 1.32% of the data


In [6]:
print("Duplicate user_ids:", data['user_id'].duplicated().sum())
# Look at one to understand it
dupe_ids = data[data['user_id'].duplicated(keep=False)]['user_id'].unique()
print("Example duplicated user:")
print(data[data['user_id'] == dupe_ids[0]])

Duplicate user_ids: 3894
Example duplicated user:
        user_id                   timestamp      group landing_page  converted
22       767017  2017-01-12 22:58:14.991443    control     new_page          0
277989   767017  2017-01-08 01:31:31.456648  treatment     new_page          0


In [10]:
# Step 1: keep only rows where group and page align correctly
df_clean = data[
    ((data['group'] == 'control') & (data['landing_page'] == 'old_page')) |
    ((data['group'] == 'treatment') & (data['landing_page'] == 'new_page'))
].copy()

# Step 2: remove users assigned to MORE THAN ONE group (ambiguous, can't attribute behavior)
conflicting_users = df_clean.groupby('user_id')['group'].nunique()
conflicting_users = conflicting_users[conflicting_users > 1].index
print("Users in conflicting groups:", len(conflicting_users))
df_clean = df_clean[~df_clean['user_id'].isin(conflicting_users)].copy()

# Step 3: drop any remaining exact duplicate users (same user appearing twice in same group)
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')

print("\nRows before:", len(data), "| Rows after cleaning:", len(df_clean))
print("\nClean group sizes:")
print(df_clean['group'].value_counts())

Users in conflicting groups: 0

Rows before: 294478 | Rows after cleaning: 290584

Clean group sizes:
group
treatment    145310
control      145274
Name: count, dtype: int64


In [11]:
from scipy.stats import chisquare

counts = df_clean['group'].value_counts().values
srm_stat, srm_p = chisquare(counts)
print(f"SRM chi-square p-value: {srm_p:.4f}")
print("Split looks healthy (p > 0.01)" if srm_p > 0.01 else "Possible SRM, investigate the split")

SRM chi-square p-value: 0.9468
Split looks healthy (p > 0.01)


### Data Quality Summary

Before analyzing results, I validated the experiment data and found two issues:

- **1.32% of rows (3,893) were mismatched** — users whose assigned group didn't match the page they actually saw (e.g., a "control" user shown the new page). These rows can't be trusted, since the group label doesn't reflect the real experience, so I removed them.
- **3,894 duplicate users** appeared in the data, some assigned to both groups. I kept only each user's first record to avoid double-counting and cross-contamination.

After cleaning, the dataset has **290,584 rows** with a near-even split (145,310 treatment / 145,274 control). A **Sample Ratio Mismatch check (chi-square p = 0.95)** confirms the split is statistically balanced, so the groups are sound to compare.

**Why it matters:** a contaminated control group biases the entire result. Cleaning first means any lift I measure later reflects the page change, not a data artifact.

In [12]:
# Conversion rate by group
conv_rates = df_clean.groupby('group')['converted'].agg(['mean', 'sum', 'count'])
conv_rates.columns = ['conversion_rate', 'conversions', 'users']
print(conv_rates)

control_rate = df_clean[df_clean['group']=='control']['converted'].mean()
treatment_rate = df_clean[df_clean['group']=='treatment']['converted'].mean()
print(f"\nControl:   {control_rate:.4f} ({control_rate*100:.2f}%)")
print(f"Treatment: {treatment_rate:.4f} ({treatment_rate*100:.2f}%)")
print(f"Absolute difference: {(treatment_rate-control_rate)*100:.3f} percentage points")

           conversion_rate  conversions   users
group                                          
control           0.120386        17489  145274
treatment         0.118808        17264  145310

Control:   0.1204 (12.04%)
Treatment: 0.1188 (11.88%)
Absolute difference: -0.158 percentage points


In [14]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Minimum detectable effect: say we'd care about a 1% relative lift over control
mde_relative = 0.01
p1 = control_rate
p2 = control_rate * (1 + mde_relative)
effect_size = proportion_effectsize(p2, p1)

analysis = NormalIndPower()
n_per_group = df_clean['group'].value_counts().min()
achieved_power = analysis.power(effect_size=effect_size, nobs1=n_per_group,
                                 alpha=0.05, ratio=1.0)
print(f"Users per group: {n_per_group:,}")
print(f"Achieved power to detect a 1% relative lift: {achieved_power:.3f}")

Users per group: 145,274
Achieved power to detect a 1% relative lift: 0.169


In [15]:
from statsmodels.stats.proportion import proportions_ztest

conversions = df_clean.groupby('group')['converted'].sum().values  # [control, treatment] alphabetical
n_obs = df_clean.groupby('group')['converted'].count().values
# groupby sorts alphabetically: control first, treatment second
control_conv, treatment_conv = conversions[0], conversions[1]
control_n, treatment_n = n_obs[0], n_obs[1]

# Test: is treatment conversion DIFFERENT from control? (two-sided)
count = np.array([treatment_conv, control_conv])
nobs = np.array([treatment_n, control_n])
z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print("Result:", "Statistically significant (p < 0.05)" if p_value < 0.05 else "NOT statistically significant (p >= 0.05)")

Z-statistic: -1.3109
P-value: 0.1899
Result: NOT statistically significant (p >= 0.05)


In [16]:
from statsmodels.stats.proportion import confint_proportions_2indep

diff_ci_low, diff_ci_upp = confint_proportions_2indep(
    treatment_conv, treatment_n, control_conv, control_n, method='wald'
)
print(f"Treatment - Control difference: {(treatment_rate-control_rate)*100:.3f} pp")
print(f"95% CI: [{diff_ci_low*100:.3f} pp, {diff_ci_upp*100:.3f} pp]")

Treatment - Control difference: -0.158 pp
95% CI: [-0.394 pp, 0.078 pp]


### Results & Recommendation

**Question:** Does the new landing page improve conversion vs. the old page?

| Group | Conversion Rate | Users |
|-------|----------------|-------|
| Control (old page) | 12.04% | 145,274 |
| Treatment (new page) | 11.88% | 145,310 |

**Finding:** The new page converted *slightly lower* (-0.16 pp), but this difference is **not statistically significant** (two-proportion z-test, p = 0.19). The 95% confidence interval on the difference [-0.39 pp, +0.08 pp] crosses zero, so we cannot conclude the new page performs differently from the old one.

With ~145K users per group, the test had ample power to detect any commercially meaningful lift. The absence of a significant effect is therefore a real result, not a sample-size limitation: the new page simply does not move conversion.

**Recommendation:** Do **not** ship the new page. There is no evidence it improves conversion, and the point estimate trends slightly negative. Keeping the existing page avoids the engineering and risk cost of a change that delivers no measurable benefit. If the team still believes in the new design, the next step would be to diagnose *why* it underperformed (e.g., segment-level analysis, or a redesigned variant) rather than rolling it out broadly.

**Why this matters:** A rigorous "no" is as valuable as a "yes." Shipping changes that don't work erodes trust in experimentation and wastes engineering cycles; this analysis protects against that.